# Complete Analysis: Brute Force vs Efficient Solution

This notebook contains comprehensive comparison tests and growth analysis for the Dutch Merchant Problem algorithms.


In [1]:
# Imports and Setup
import sys
import os
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
from pathlib import Path

# Add project root to path
project_root = Path.cwd()
sys.path.append(str(project_root))

# Import algorithms
from pure_brute_force import pure_brute_force
from pruning_brute_force import brute_force_with_pruning
from BF_DP_hybrid import hybrid_brute_force_dp
from efficient_solution import two_phase_hybrid_solve, generate_multi_item_instance
from instance_generator import generate_unified_test_instance
from comparison_test import convert_single_item_to_multi_item

# Setup directories
data_dir = project_root / 'data'
figures_dir = project_root / 'figures'
data_dir.mkdir(exist_ok=True)
figures_dir.mkdir(exist_ok=True)

print("Setup complete!")
print(f"Data directory: {data_dir}")
print(f"Figures directory: {figures_dir}")


Setup complete!
Data directory: /home/ari/Collage/04-Forth_Year/Preimer_Semestre/DAA/El_Comerciante_Holandes/data
Figures directory: /home/ari/Collage/04-Forth_Year/Preimer_Semestre/DAA/El_Comerciante_Holandes/figures


## 1. Comparison Test: Brute Force vs Efficient Solution

Compare all brute force algorithms (pure, pruned, hybrid) with the efficient solution on small-medium instances (n=3-8) where all algorithms can complete.


In [4]:
# Comparison Test Implementation
def run_comparison_test(n_values=[3, 4, 5, 6, 7, 8], timeout=200.0, seed=42):
    """
    Run comparison test between all BF algorithms and efficient solution.
    
    Returns:
        DataFrame with comparison results
    """
    results = []
    
    for n in n_values:
        print(f"\nTesting n={n}...")
        
        # Generate single-item instance
        instance_single = generate_unified_test_instance(n, seed=seed)
        
        # Convert to multi-item format for efficient solution
        instance_multi = convert_single_item_to_multi_item(instance_single)
        
        result_row = {'n': n}
        
        # Run Pure Brute Force
        print("  Running Pure BF...", end=" ")
        try:
            start = time.time()
            pure_result = pure_brute_force(instance_single, timeout=timeout)
            pure_time = time.time() - start
            result_row['pure_bf_time'] = pure_time
            result_row['pure_bf_capital'] = pure_result.get('final_capital')
            result_row['pure_bf_solutions'] = pure_result.get('explored_solutions', 0)
            print(f"{pure_time:.3f}s")
        except Exception as e:
            print(f"Error: {e}")
            result_row['pure_bf_time'] = float('inf')
            result_row['pure_bf_capital'] = None
            result_row['pure_bf_solutions'] = 0
        
        # Run Pruned Brute Force
        print("  Running Pruned BF...", end=" ")
        try:
            start = time.time()
            pruned_result = brute_force_with_pruning(instance_single, timeout=timeout)
            pruned_time = time.time() - start
            result_row['pruned_bf_time'] = pruned_time
            result_row['pruned_bf_capital'] = pruned_result.get('final_capital')
            stats = pruned_result.get('pruning_statistics', {})
            result_row['pruned_bf_nodes'] = stats.get('branches_explored', 0)
            print(f"{pruned_time:.3f}s")
        except Exception as e:
            print(f"Error: {e}")
            result_row['pruned_bf_time'] = float('inf')
            result_row['pruned_bf_capital'] = None
            result_row['pruned_bf_nodes'] = 0
        
        # Run Hybrid BF (BF + DP)
        print("  Running Hybrid BF...", end=" ")
        try:
            start = time.time()
            hybrid_result = hybrid_brute_force_dp(instance_single, timeout=timeout)
            hybrid_time = time.time() - start
            result_row['hybrid_bf_time'] = hybrid_time
            result_row['hybrid_bf_capital'] = hybrid_result.get('final_capital')
            hybrid_stats = hybrid_result.get('statistics', {})
            result_row['hybrid_bf_routes'] = hybrid_stats.get('routes_explored', 0)
            result_row['hybrid_bf_dp_execs'] = hybrid_stats.get('dp_executions', 0)
            print(f"{hybrid_time:.3f}s")
        except Exception as e:
            print(f"Error: {e}")
            result_row['hybrid_bf_time'] = float('inf')
            result_row['hybrid_bf_capital'] = None
            result_row['hybrid_bf_routes'] = 0
            result_row['hybrid_bf_dp_execs'] = 0
        
        # Run Efficient Solution
        print("  Running Efficient Solution...", end=" ")
        try:
            start = time.time()
            efficient_result = two_phase_hybrid_solve(instance_multi, timeout=timeout, beam_width=100)
            efficient_time = time.time() - start
            result_row['efficient_time'] = efficient_time
            result_row['efficient_capital'] = efficient_result.get('capital')
            result_row['efficient_routes_gen'] = efficient_result.get('routes_generated', 0)
            result_row['efficient_routes_eval'] = efficient_result.get('routes_evaluated', 0)
            print(f"{efficient_time:.3f}s")
        except Exception as e:
            print(f"Error: {e}")
            result_row['efficient_time'] = float('inf')
            result_row['efficient_capital'] = None
            result_row['efficient_routes_gen'] = 0
            result_row['efficient_routes_eval'] = 0
        
        # Verify capital matches (within tolerance)
        capitals = [
            result_row.get('pure_bf_capital'),
            result_row.get('pruned_bf_capital'),
            result_row.get('hybrid_bf_capital'),
            result_row.get('efficient_capital')
        ]
        valid_capitals = [c for c in capitals if c is not None]
        
        if len(valid_capitals) > 1:
            max_cap = max(valid_capitals)
            min_cap = min(valid_capitals)
            tolerance = 0.01  # Small tolerance for floating point
            result_row['capital_match'] = (max_cap - min_cap) < tolerance
        else:
            result_row['capital_match'] = None
        
        results.append(result_row)
        print(f"  Capital match: {result_row['capital_match']}")
    
    return pd.DataFrame(results)

# Run the comparison test
print("Starting comparison test...")
comparison_df = run_comparison_test(n_values=[2, 3, 4, 5, 6, 7, 8], timeout=200.0)
print("\nComparison test complete!")
print(comparison_df)


Starting comparison test...

Testing n=2...
  Running Pure BF... 0.000s
  Running Pruned BF... 0.000s
  Running Hybrid BF... 0.000s
  Running Efficient Solution... 0.000s
  Capital match: True

Testing n=3...
  Running Pure BF... 0.000s
  Running Pruned BF... 0.000s
  Running Hybrid BF... 0.000s
  Running Efficient Solution... 0.000s
  Capital match: True

Testing n=4...
  Running Pure BF... 0.001s
  Running Pruned BF... 0.000s
  Running Hybrid BF... 0.000s
  Running Efficient Solution... 0.001s
  Capital match: True

Testing n=5...
  Running Pure BF... 0.004s
  Running Pruned BF... 0.005s
  Running Hybrid BF... 0.002s
  Running Efficient Solution... 0.002s
  Capital match: False

Testing n=6...
  Running Pure BF... 0.060s
  Running Pruned BF... 0.016s
  Running Hybrid BF... 0.014s
  Running Efficient Solution... 0.009s
  Capital match: False

Testing n=7...
  Running Pure BF... 1.130s
  Running Pruned BF... 0.315s
  Running Hybrid BF... 0.334s
  Running Efficient Solution... 0.049s
  

In [5]:
# Save comparison results
comparison_df.to_csv(data_dir / 'comparison_bf_vs_efficient.csv', index=False)
print("Comparison results saved to data/comparison_bf_vs_efficient.csv")
comparison_df


Comparison results saved to data/comparison_bf_vs_efficient.csv


,n,pure_bf_time,pure_bf_capital,pure_bf_solutions,pruned_bf_time,pruned_bf_capital,pruned_bf_nodes,hybrid_bf_time,hybrid_bf_capital,hybrid_bf_routes,hybrid_bf_dp_execs,efficient_time,efficient_capital,efficient_routes_gen,efficient_routes_eval,capital_match
0,2,0.000046,27,4,0.000038,27,4,0.000036,27,2,0,0.000124,NaN,2,2,True
1,3,0.000060,78,25,0.000080,78,23,0.000069,78,5,0,0.000433,78.0,5,5,True
2,4,0.000571,89,226,0.000377,89,184,0.000342,89,16,6,0.000555,89.0,15,7,True
3,5,0.004426,116,2713,0.005187,116,1821,0.002107,116,65,48,0.002192,111.0,48,19,False
4,6,0.059930,139,40696,0.015768,139,21776,0.014115,139,326,300,0.009150,121.0,215,41,False
5,7,1.130060,176,732529,0.314735,176,313181,0.334213,176,1957,1679,0.049069,176.0,588,42,True
6,8,30.790415,184,15383110,5.211610,184,4646872,0.415983,183,13700,3590,0.139375,160.0,993,93,False


## 2. Growth Analysis with m (Number of Items)

Analyze how the efficient solution scales with the number of item types m, with k = capacity (B).


In [ ]:
# Growth Analysis with m
def run_growth_analysis_m(n_values=[10, 15], m_values=[1, 2, 3, 4, 5, 6, 8, 10], timeout=6000.0, seed=42):
    """
    Analyze growth of efficient solution with varying m (number of items).
    For each m, k is set to capacity (B).
    
    Returns:
        DataFrame with growth analysis results
    """
    results = []
    
    for n in n_values:
        print(f"\nTesting n={n}...")
        
        for m in m_values:
            print(f"  Testing m={m}...", end=" ")
            
            # Generate multi-item instance
            # First generate to get capacity, then use that as k
            instance = generate_multi_item_instance(n_ports=n, m_items=m, k_max_units=1, seed=seed)
            capacity = instance['capacity']
            
            # Set k = capacity
            instance['max_units_per_op'] = capacity
            
            # Run efficient solution
            try:
                start = time.time()
                result = two_phase_hybrid_solve(instance, timeout=timeout, beam_width=200)
                exec_time = time.time() - start
                
                results.append({
                    'n': n,
                    'm': m,
                    'k': capacity,
                    'execution_time': exec_time,
                    'routes_generated': result.get('routes_generated', 0),
                    'routes_evaluated': result.get('routes_evaluated', 0),
                    'final_capital': result.get('capital'),
                    'timeout': result.get('timeout', False)
                })
                print(f"{exec_time:.3f}s")
            except Exception as e:
                print(f"Error: {e}")
                results.append({
                    'n': n,
                    'm': m,
                    'k': capacity,
                    'execution_time': float('inf'),
                    'routes_generated': 0,
                    'routes_evaluated': 0,
                    'final_capital': None,
                    'timeout': True
                })
    
    return pd.DataFrame(results)

# Run growth analysis
print("Starting growth analysis with m...")
growth_m_df = run_growth_analysis_m(n_values=[15, 20, 25, 30], m_values=[2, 6, 10, 12, 14, 15])
print("\nGrowth analysis complete!")
print(growth_m_df)


In [ ]:
# Save growth analysis results
growth_m_df.to_csv(data_dir / 'growth_analysis_m.csv', index=False)
print("Growth analysis results saved to data/growth_analysis_m.csv")
growth_m_df


## 3. Scalability Analysis with n

Analyze how the efficient solution scales with the number of ports n (with m=1, k=1).


In [6]:
# Scalability Analysis with n
def run_scalability_analysis(n_values=[10, 15, 20, 25, 30], timeout=6000.0, seed=42):
    """
    Analyze scalability of efficient solution with varying n (number of ports).
    Uses m=1, k=1 for all tests.
    
    Returns:
        DataFrame with scalability results
    """
    results = []
    
    for n in n_values:
        print(f"Testing n={n}...", end=" ")
        
        # Generate instance with m=1, k=1
        instance = generate_multi_item_instance(n_ports=n, m_items=2, k_max_units=1, seed=seed)
        capacity = instance.get('capacity')
        instance['max_units_per_op'] = capacity
        
        # Run efficient solution
        try:
            start = time.time()
            result = two_phase_hybrid_solve(instance, timeout=timeout, beam_width=200)
            exec_time = time.time() - start
            
            results.append({
                'n': n,
                'm': 1,
                'k': capacity,
                'execution_time': exec_time,
                'routes_generated': result.get('routes_generated', 0),
                'routes_evaluated': result.get('routes_evaluated', 0),
                'initial_capital': instance.get('initial_capital'),
                'final_capital': result.get('capital'),
                'timeout': result.get('timeout', False)
            })
            print(f"{exec_time:.3f}s")
        except Exception as e:
            print(f"Error: {e}")
            results.append({
                'n': n,
                'm': 1,
                'k': capacity,
                'execution_time': float('inf'),
                'routes_generated': 0,
                'routes_evaluated': 0,
                'initial_capital': None,
                'final_capital': None,
                'timeout': True
            })
    
    return pd.DataFrame(results)

# Run scalability analysis
print("Starting scalability analysis...")
scalability_df = run_scalability_analysis(n_values=[10, 15, 20, 25, 30])
print("\nScalability analysis complete!")
print(scalability_df)


Starting scalability analysis...
Testing n=10... 1.679s
Testing n=15... 0.442s
Testing n=20... 0.928s
Testing n=25... 1.556s
Testing n=30... 2.996s

Scalability analysis complete!
    n  m  k  execution_time  routes_generated  routes_evaluated  \
0  10  1  5        1.678802              1291                18   
1  15  1  5        0.441700              2357                 1   
2  20  1  5        0.928291              3420                 1   
3  25  1  5        1.556093              4425                 1   
4  30  1  5        2.995719              5430                 1   

   initial_capital  final_capital  timeout  
0              240          742.0    False  
1              340          936.4    False  
2              440         1370.8    False  
3              540         1595.4    False  
4              640         1898.8    False  


In [7]:
# Save scalability results
scalability_df.to_csv(data_dir / 'efficient_scalability.csv', index=False)
print("Scalability results saved to data/efficient_scalability.csv")
scalability_df


Scalability results saved to data/efficient_scalability.csv


,n,m,k,execution_time,routes_generated,routes_evaluated,initial_capital,final_capital,timeout
0,10,1,5,1.678802,1291,18,240,742.0,False
1,15,1,5,0.441700,2357,1,340,936.4,False
2,20,1,5,0.928291,3420,1,440,1370.8,False
3,25,1,5,1.556093,4425,1,540,1595.4,False
4,30,1,5,2.995719,5430,1,640,1898.8,False


## 4. Beam Width Sensitivity Analysis

Analyze how different beam width values affect the efficient solution's performance.


In [8]:
# Beam Width Sensitivity Analysis
def run_beam_width_analysis(n=15, beam_widths=[50, 100, 200, 500], timeout=300.0, seed=42):
    """
    Analyze sensitivity of efficient solution to beam width parameter.
    
    Returns:
        DataFrame with beam width analysis results
    """
    results = []
    
    # Generate a fixed instance
    instance = generate_multi_item_instance(n_ports=n, m_items=1, k_max_units=1, seed=seed)
    
    for beam_width in beam_widths:
        print(f"Testing beam_width={beam_width}...", end=" ")
        
        try:
            start = time.time()
            result = two_phase_hybrid_solve(instance, timeout=timeout, beam_width=beam_width)
            exec_time = time.time() - start
            
            results.append({
                'n': n,
                'beam_width': beam_width,
                'execution_time': exec_time,
                'routes_generated': result.get('routes_generated', 0),
                'routes_evaluated': result.get('routes_evaluated', 0),
                'final_capital': result.get('capital'),
                'timeout': result.get('timeout', False)
            })
            print(f"{exec_time:.3f}s")
        except Exception as e:
            print(f"Error: {e}")
            results.append({
                'n': n,
                'beam_width': beam_width,
                'execution_time': float('inf'),
                'routes_generated': 0,
                'routes_evaluated': 0,
                'final_capital': None,
                'timeout': True
            })
    
    return pd.DataFrame(results)

# Run beam width analysis
print("Starting beam width sensitivity analysis...")
beam_width_df = run_beam_width_analysis(n=15, beam_widths=[50, 100, 200, 500])
print("\nBeam width analysis complete!")
print(beam_width_df)


Starting beam width sensitivity analysis...
Testing beam_width=50... 0.076s
Testing beam_width=100... 0.243s
Testing beam_width=200... 0.311s
Testing beam_width=500... 0.468s

Beam width analysis complete!
    n  beam_width  execution_time  routes_generated  routes_evaluated  \
0  15          50        0.075988               572                 1   
1  15         100        0.243231              1063                 1   
2  15         200        0.311078              1728                 1   
3  15         500        0.468158              3139                 1   

   final_capital  timeout  
0          430.2    False  
1          430.2    False  
2          430.2    False  
3          430.2    False  


In [ ]:
# Save beam width analysis results
beam_width_df.to_csv(data_dir / 'beam_width_analysis.csv', index=False)
print("Beam width analysis results saved to data/beam_width_analysis.csv")
beam_width_df


## 5. Visualizations

Create visualizations for all analyses.


In [ ]:
# Visualization: Comparison Test
try:
    plt.style.use('seaborn-v0_8-darkgrid')
except:
    try:
        plt.style.use('seaborn-darkgrid')
    except:
        plt.style.use('default')
fig, ax = plt.subplots(figsize=(12, 6))

# Prepare data
n_vals = comparison_df['n'].values
pure_times = comparison_df['pure_bf_time'].replace([float('inf'), np.inf], np.nan)
pruned_times = comparison_df['pruned_bf_time'].replace([float('inf'), np.inf], np.nan)
hybrid_times = comparison_df['hybrid_bf_time'].replace([float('inf'), np.inf], np.nan)
efficient_times = comparison_df['efficient_time'].replace([float('inf'), np.inf], np.nan)

x = np.arange(len(n_vals))
width = 0.2

ax.bar(x - 1.5*width, pure_times, width, label='Pure BF', alpha=0.8)
ax.bar(x - 0.5*width, pruned_times, width, label='Pruned BF', alpha=0.8)
ax.bar(x + 0.5*width, hybrid_times, width, label='Hybrid BF', alpha=0.8)
ax.bar(x + 1.5*width, efficient_times, width, label='Efficient', alpha=0.8)

ax.set_xlabel('Number of Ports (n)', fontsize=12)
ax.set_ylabel('Execution Time (seconds)', fontsize=12)
ax.set_title('Algorithm Comparison: Execution Time', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(n_vals)
ax.legend()
ax.set_yscale('log')

plt.tight_layout()
plt.savefig(figures_dir / 'comparison_bf_vs_efficient.png', dpi=300, bbox_inches='tight')
plt.savefig(figures_dir / 'comparison_bf_vs_efficient.pdf', bbox_inches='tight')
plt.show()

print("Comparison visualization saved!")


In [ ]:
# Visualization: Growth Analysis with m
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Filter out infinite values
growth_clean = growth_m_df[growth_m_df['execution_time'] != float('inf')].copy()

for idx, n_val in enumerate([10, 15]):
    if n_val not in growth_clean['n'].values:
        continue
    
    n_data = growth_clean[growth_clean['n'] == n_val]
    
    # Execution time vs m
    axes[0, idx].plot(n_data['m'], n_data['execution_time'], 'o-', linewidth=2, markersize=8)
    axes[0, idx].set_xlabel('Number of Items (m)', fontsize=11)
    axes[0, idx].set_ylabel('Execution Time (seconds)', fontsize=11)
    axes[0, idx].set_title(f'Execution Time vs m (n={n_val})', fontsize=12, fontweight='bold')
    axes[0, idx].grid(True, alpha=0.3)
    axes[0, idx].set_yscale('log')
    
    # Routes generated vs m
    axes[1, idx].plot(n_data['m'], n_data['routes_generated'], 's-', linewidth=2, markersize=8, color='orange')
    axes[1, idx].set_xlabel('Number of Items (m)', fontsize=11)
    axes[1, idx].set_ylabel('Routes Generated', fontsize=11)
    axes[1, idx].set_title(f'Routes Generated vs m (n={n_val})', fontsize=12, fontweight='bold')
    axes[1, idx].grid(True, alpha=0.3)
    axes[1, idx].set_yscale('log')

plt.tight_layout()
plt.savefig(figures_dir / 'growth_analysis_m.png', dpi=300, bbox_inches='tight')
plt.savefig(figures_dir / 'growth_analysis_m.pdf', bbox_inches='tight')
plt.show()

print("Growth analysis visualization saved!")


In [ ]:
# Visualization: Scalability Analysis
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Filter out infinite values
scalability_clean = scalability_df[scalability_df['execution_time'] != float('inf')].copy()

# Execution time vs n
axes[0].plot(scalability_clean['n'], scalability_clean['execution_time'], 'o-', linewidth=2, markersize=10, color='blue')
axes[0].set_xlabel('Number of Ports (n)', fontsize=12)
axes[0].set_ylabel('Execution Time (seconds)', fontsize=12)
axes[0].set_title('Scalability: Execution Time vs n', fontsize=13, fontweight='bold')
axes[0].grid(True, alpha=0.3)
axes[0].set_yscale('log')

# Routes generated vs n
axes[1].plot(scalability_clean['n'], scalability_clean['routes_generated'], 's-', linewidth=2, markersize=10, color='green')
axes[1].set_xlabel('Number of Ports (n)', fontsize=12)
axes[1].set_ylabel('Routes Generated', fontsize=12)
axes[1].set_title('Scalability: Routes Generated vs n', fontsize=13, fontweight='bold')
axes[1].grid(True, alpha=0.3)
axes[1].set_yscale('log')

# Routes evaluated vs n
axes[2].plot(scalability_clean['n'], scalability_clean['routes_evaluated'], '^-', linewidth=2, markersize=10, color='red')
axes[2].set_xlabel('Number of Ports (n)', fontsize=12)
axes[2].set_ylabel('Routes Evaluated', fontsize=12)
axes[2].set_title('Scalability: Routes Evaluated vs n', fontsize=13, fontweight='bold')
axes[2].grid(True, alpha=0.3)
axes[2].set_yscale('log')

plt.tight_layout()
plt.savefig(figures_dir / 'efficient_scalability.png', dpi=300, bbox_inches='tight')
plt.savefig(figures_dir / 'efficient_scalability.pdf', bbox_inches='tight')
plt.show()

print("Scalability visualization saved!")


In [ ]:
# Visualization: Beam Width Sensitivity
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Filter out infinite values
beam_clean = beam_width_df[beam_width_df['execution_time'] != float('inf')].copy()

# Execution time vs beam width
axes[0].plot(beam_clean['beam_width'], beam_clean['execution_time'], 'o-', linewidth=2, markersize=10, color='purple')
axes[0].set_xlabel('Beam Width', fontsize=12)
axes[0].set_ylabel('Execution Time (seconds)', fontsize=12)
axes[0].set_title('Beam Width Sensitivity: Execution Time', fontsize=13, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Routes generated vs beam width
axes[1].plot(beam_clean['beam_width'], beam_clean['routes_generated'], 's-', linewidth=2, markersize=10, color='orange')
axes[1].set_xlabel('Beam Width', fontsize=12)
axes[1].set_ylabel('Routes Generated', fontsize=12)
axes[1].set_title('Beam Width Sensitivity: Routes Generated', fontsize=13, fontweight='bold')
axes[1].grid(True, alpha=0.3)

# Routes evaluated vs beam width
axes[2].plot(beam_clean['beam_width'], beam_clean['routes_evaluated'], '^-', linewidth=2, markersize=10, color='teal')
axes[2].set_xlabel('Beam Width', fontsize=12)
axes[2].set_ylabel('Routes Evaluated', fontsize=12)
axes[2].set_title('Beam Width Sensitivity: Routes Evaluated', fontsize=13, fontweight='bold')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(figures_dir / 'beam_width_analysis.png', dpi=300, bbox_inches='tight')
plt.savefig(figures_dir / 'beam_width_analysis.pdf', bbox_inches='tight')
plt.show()

print("Beam width analysis visualization saved!")


## 6. Key Findings and Summary


In [ ]:
# Generate summary statistics
print("="*80)
print("KEY FINDINGS SUMMARY")
print("="*80)

print("\n1. COMPARISON TEST RESULTS:")
print("-" * 80)
if not comparison_df.empty:
    print(f"   • Tested instances: n = {', '.join(map(str, comparison_df['n'].values))}")
    print(f"   • Capital matches: {comparison_df['capital_match'].sum()} / {comparison_df['capital_match'].notna().sum()}")
    
    # Find best algorithm for each n
    for n_val in comparison_df['n'].values:
        row = comparison_df[comparison_df['n'] == n_val].iloc[0]
        times = {
            'Pure BF': row['pure_bf_time'] if row['pure_bf_time'] != float('inf') else np.inf,
            'Pruned BF': row['pruned_bf_time'] if row['pruned_bf_time'] != float('inf') else np.inf,
            'Hybrid BF': row['hybrid_bf_time'] if row['hybrid_bf_time'] != float('inf') else np.inf,
            'Efficient': row['efficient_time'] if row['efficient_time'] != float('inf') else np.inf
        }
        best = min(times, key=times.get)
        print(f"   • n={n_val}: Fastest = {best} ({times[best]:.3f}s)")

print("\n2. GROWTH ANALYSIS WITH m:")
# print("-" * 80)
# if not growth_m_df.empty:
#     growth_clean = growth_m_df[growth_m_df['execution_time'] != float('inf')]
#     if not growth_clean.empty:
#         for n_val in growth_clean['n'].unique():
#             n_data = growth_clean[growth_clean['n'] == n_val]
#             if len(n_data) > 1:
#                 m_min = n_data['m'].min()
#                 m_max = n_data['m'].max()
#                 time_min = n_data[n_data['m'] == m_min]['execution_time'].values[0]
#                 time_max = n_data[n_data['m'] == m_max]['execution_time'].values[0]
#                 growth_factor = time_max / time_min if time_min > 0 else np.inf
#                 print(f"   • n={n_val}: Time growth from m={m_min} to m={m_max}: {growth_factor:.2f}x")

print("\n3. SCALABILITY ANALYSIS:")
# print("-" * 80)
# if not scalability_df.empty:
#     scalability_clean = scalability_df[scalability_df['execution_time'] != float('inf')]
#     if not scalability_clean.empty and len(scalability_clean) > 1:
#         n_min = scalability_clean['n'].min()
#         n_max = scalability_clean['n'].max()
#         time_min = scalability_clean[scalability_clean['n'] == n_min]['execution_time'].values[0]
#         time_max = scalability_clean[scalability_clean['n'] == n_max]['execution_time'].values[0]
#         n_factor = n_max / n_min
#         time_factor = time_max / time_min if time_min > 0 else np.inf
#         print(f"   • Tested range: n={n_min} to n={n_max} ({n_factor:.1f}x increase)")
#         print(f"   • Time increase: {time_factor:.2f}x")
#         print(f"   • Effective complexity: ~O(n^{np.log(time_factor)/np.log(n_factor):.2f})")

print("\n4. BEAM WIDTH SENSITIVITY:")
print("-" * 80)
if not beam_width_df.empty:
    beam_clean = beam_width_df[beam_width_df['execution_time'] != float('inf')]
    if not beam_clean.empty and len(beam_clean) > 1:
        bw_min = beam_clean['beam_width'].min()
        bw_max = beam_clean['beam_width'].max()
        routes_min = beam_clean[beam_clean['beam_width'] == bw_min]['routes_generated'].values[0]
        routes_max = beam_clean[beam_clean['beam_width'] == bw_max]['routes_generated'].values[0]
        print(f"   • Beam width range: {bw_min} to {bw_max}")
        print(f"   • Routes generated: {routes_min} to {routes_max} ({routes_max/routes_min:.2f}x increase)")
        print(f"   • Impact: Larger beam width generates more routes but may not always improve solution quality")

print("\n" + "="*80)


## 7. Debugging: Why Efficient Solution Differs from BF

Analyzing why the efficient solution returns different (lower) capital values than the brute force algorithms.


In [ ]:
# Debug: Compare a specific instance
n_test = 3
instance_single = generate_unified_test_instance(n_test, seed=42)
instance_multi = convert_single_item_to_multi_item(instance_single)

print("Instance details:")
print(f"  Initial capital: {instance_single['initial_capital']}")
print(f"  Capacity: {instance_single['capacity']}")
print(f"  Max time: {instance_single['max_time']}")
print(f"  Purchase prices: {instance_single['purchase_prices']}")
print(f"  Sale prices: {instance_single['sale_prices']}")

# Run BF
pure_result = pure_brute_force(instance_single, timeout=200.0)
print(f"\nPure BF result:")
print(f"  Capital: {pure_result.get('final_capital')}")
print(f"  Route: {pure_result.get('optimal_tour')}")
print(f"  Decisions: {pure_result.get('optimal_decisions')}")

# Run Efficient
efficient_result = two_phase_hybrid_solve(instance_multi, timeout=200.0, beam_width=100)
print(f"\nEfficient result:")
print(f"  Capital: {efficient_result.get('capital')}")
print(f"  Route: {efficient_result.get('route')}")
print(f"  Routes generated: {efficient_result.get('routes_generated')}")
print(f"  Routes evaluated: {efficient_result.get('routes_evaluated')}")
print(f"  Decisions: {efficient_result.get('decisions')}")

# Check if routes match
if pure_result.get('optimal_tour') == efficient_result.get('route'):
    print("\n✓ Routes match!")
else:
    print(f"\n✗ Routes differ!")
    print(f"  BF route: {pure_result.get('optimal_tour')}")
    print(f"  Efficient route: {efficient_result.get('route')}")


## 8. Improvements Made to Efficient Solution

Based on analysis of comparison results and research on beam search improvements, the following enhancements were implemented:


In [ ]:
# Test the improvements
print("Testing improved efficient solution...")
print("="*80)

# Test on a failing case (n=5)
n_test = 5
instance_single = generate_unified_test_instance(n_test, seed=42)
instance_multi = convert_single_item_to_multi_item(instance_single)

print(f"\nInstance: n={n_test}")
print(f"  Initial capital: {instance_single['initial_capital']}")
print(f"  Capacity: {instance_single['capacity']}")

# Run BF
pure_result = pure_brute_force(instance_single, timeout=200.0)
bf_capital = pure_result.get('final_capital')

# Run Improved Efficient
efficient_result = two_phase_hybrid_solve(instance_multi, timeout=200.0, beam_width=100)
eff_capital = efficient_result.get('capital')

print(f"\nResults:")
print(f"  BF Capital: {bf_capital}")
print(f"  Efficient Capital: {eff_capital}")
print(f"  Difference: {abs(bf_capital - eff_capital) if (bf_capital and eff_capital) else 'N/A'}")
print(f"  Routes Generated: {efficient_result.get('routes_generated')}")
print(f"  Routes Evaluated: {efficient_result.get('routes_evaluated')}")
print(f"  Match: {abs(bf_capital - eff_capital) < 0.01 if (bf_capital and eff_capital) else False}")

if eff_capital and eff_capital < instance_single['initial_capital']:
    print(f"\n⚠️  WARNING: Efficient solution returned capital below initial budget!")
else:
    print(f"\n✓ Solution is valid (capital >= initial budget)")


In [ ]:
# Quick test to verify fixes work
print("Testing route generation fixes...")
print("="*80)

# Test n=2 (should generate routes)
n_test = 2
instance_single = generate_unified_test_instance(n_test, seed=42)
instance_multi = convert_single_item_to_multi_item(instance_single)

print(f"\nTest 1: n={n_test}")
from efficient_solution import generate_promising_routes
routes = generate_promising_routes(instance_multi, beam_width=10, timeout=10.0)
print(f"  Routes generated: {len(routes)}")
if routes:
    print(f"  Sample route: {routes[0][0]}, bound: {routes[0][1]:.2f}")
else:
    print("  ⚠️  No routes generated!")

# Test n=3
n_test = 3
instance_single = generate_unified_test_instance(n_test, seed=42)
instance_multi = convert_single_item_to_multi_item(instance_single)

print(f"\nTest 2: n={n_test}")
routes = generate_promising_routes(instance_multi, beam_width=10, timeout=10.0)
print(f"  Routes generated: {len(routes)}")
if routes:
    print(f"  Top 3 routes:")
    for i, (route, bound) in enumerate(routes[:3]):
        print(f"    {i+1}. Route: {route}, Bound: {bound:.2f}")

# Test full solve
print(f"\nTest 3: Full solve for n={n_test}")
efficient_result = two_phase_hybrid_solve(instance_multi, timeout=200.0, beam_width=100)
print(f"  Capital: {efficient_result.get('capital')}")
print(f"  Routes generated: {efficient_result.get('routes_generated')}")
print(f"  Routes evaluated: {efficient_result.get('routes_evaluated')}")
print(f"  Route: {efficient_result.get('route')}")

if efficient_result.get('routes_generated', 0) == 0:
    print("\n  ⚠️  WARNING: Still no routes generated!")
else:
    print("\n  ✓ Routes are being generated successfully!")


## 5. Beam Width Capital Analysis Visualization

Generate a figure showing capital obtained vs beam width parameter for inclusion in the report.


In [ ]:
# Load beam width analysis data
beam_width_df = pd.read_csv(data_dir / 'beam_width_analysis.csv')

# Create dual-axis plot: Capital and Time vs Beam Width
fig, ax1 = plt.subplots(figsize=(10, 6))

# Left axis: Capital
color1 = 'steelblue'
ax1.set_xlabel('Beam Width', fontsize=12, fontweight='bold')
ax1.set_ylabel('Final Capital', fontsize=12, fontweight='bold', color=color1)
line1 = ax1.plot(beam_width_df['beam_width'], beam_width_df['final_capital'], 
                 'o-', linewidth=2.5, markersize=12, color=color1, label='Final Capital')
ax1.tick_params(axis='y', labelcolor=color1)
ax1.grid(True, alpha=0.3, linestyle='--')
ax1.set_xticks(beam_width_df['beam_width'])

# Right axis: Execution Time
ax2 = ax1.twinx()
color2 = 'coral'
ax2.set_ylabel('Execution Time (s)', fontsize=12, fontweight='bold', color=color2)
line2 = ax2.plot(beam_width_df['beam_width'], beam_width_df['execution_time'], 
                 's--', linewidth=2.5, markersize=12, color=color2, label='Execution Time')
ax2.tick_params(axis='y', labelcolor=color2)

# Add value labels
for i, row in beam_width_df.iterrows():
    ax1.annotate(f'{row["final_capital"]:.1f}', 
                (row['beam_width'], row['final_capital']),
                textcoords="offset points", xytext=(0,10), ha='center', fontsize=10, color=color1)
    ax2.annotate(f'{row["execution_time"]:.2f}s', 
                (row['beam_width'], row['execution_time']),
                textcoords="offset points", xytext=(0,-15), ha='center', fontsize=10, color=color2)

# Title and legend
plt.title('Beam Width Sensitivity: Capital and Execution Time\n(n=15 ports)', 
          fontsize=14, fontweight='bold', pad=20)

# Combine legends
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=10)

plt.tight_layout()
plt.savefig(figures_dir / 'beam_width_capital_time.png', dpi=300, bbox_inches='tight')
plt.savefig(figures_dir / 'beam_width_capital_time.pdf', bbox_inches='tight')
print("Beam width capital-time visualization saved!")
plt.show()


## 6. Time Growth Comparison: All Four Methods

Compare execution time growth across all four algorithms (Pure BF, Pruned BF, Hybrid BF, Efficient Solution).


In [ ]:
# Load comparison data
comparison_df = pd.read_csv(data_dir / 'comparison_bf_vs_efficient.csv')

# Filter out invalid rows and prepare data
comparison_clean = comparison_df[
    (comparison_df['n'] >= 2) & 
    (comparison_df['pure_bf_time'] != float('inf')) &
    (comparison_df['pure_bf_time'].notna())
].copy()

# Create time growth comparison plot
fig, ax = plt.subplots(figsize=(12, 7))

# Plot lines for each method
n_values = comparison_clean['n'].values

# Pure Brute Force
# ax.plot(n_values, comparison_clean['pure_bf_time'], 
#         'o-', linewidth=2.5, markersize=10, label='Pure Brute Force', color='#e74c3c')

# Pruned Brute Force
ax.plot(n_values, comparison_clean['pruned_bf_time'], 
        's-', linewidth=2.5, markersize=10, label='Pruned Brute Force', color='#f39c12')

# # Hybrid Brute Force
# ax.plot(n_values, comparison_clean['hybrid_bf_time'], 
#         '^-', linewidth=2.5, markersize=10, label='Hybrid Brute Force (BF+DP)', color='#9b59b6')

# Efficient Solution
ax.plot(n_values, comparison_clean['efficient_time'], 
        'd-', linewidth=2.5, markersize=10, label='Efficient Solution', color='#2ecc71')

# Formatting
ax.set_xlabel('Number of Ports (n)', fontsize=13, fontweight='bold')
ax.set_ylabel('Execution Time (seconds)', fontsize=13, fontweight='bold')
ax.set_title('Execution Time Growth Comparison: All Four Algorithms', 
             fontsize=14, fontweight='bold', pad=15)
ax.set_yscale('log')  # Log scale for better visualization
ax.grid(True, alpha=0.3, linestyle='--', which='both')
ax.legend(loc='upper left', fontsize=11, framealpha=0.9)

# Add value labels for the last point of each method
for method, col, marker in [
#     ('pure_bf_time', '#e74c3c', 'o'),
    ('pruned_bf_time', '#f39c12', 's'),
#     ('hybrid_bf_time', '#9b59b6', '^'),
    ('efficient_time', '#2ecc71', 'd')
]:
    if col in comparison_clean.columns:
        last_idx = comparison_clean[col].last_valid_index()
        if last_idx is not None:
            last_n = comparison_clean.loc[last_idx, 'n']
            last_time = comparison_clean.loc[last_idx, col]
            ax.annotate(f'{last_time:.3f}s', 
                       (last_n, last_time),
                       textcoords="offset points", xytext=(5,5), 
                       ha='left', fontsize=9, color=col, fontweight='bold')

plt.tight_layout()
plt.savefig(figures_dir / 'time_growth_comparison_all_methods.png', dpi=300, bbox_inches='tight')
plt.savefig(figures_dir / 'time_growth_comparison_all_methods.pdf', bbox_inches='tight')
print("Time growth comparison visualization saved!")
plt.show()


In [ ]:
# Growth Analysis with m
def run_growth_analysis_m(n_values=[10, 15], m_values=[1, 2, 3, 4, 5, 6, 8, 10], timeout=600.0, seed=42):
    """
    Analyze growth of efficient solution with varying m (number of items).
    For each m, k is set to capacity (B).
    
    Returns:
        DataFrame with growth analysis results
    """
    results = []
    
    for n in n_values:
        print(f"\nTesting n={n}...")
        
        for m in m_values:
            print(f"  Testing m={m}...", end=" ")
            
            # Generate multi-item instance
            # First generate to get capacity, then use that as k
            instance = generate_multi_item_instance(n_ports=n, m_items=m, k_max_units=1, seed=seed)
            capacity = instance['capacity']
            
            # Set k = capacity
            instance['max_units_per_op'] = capacity
            
            # Run efficient solution
            try:
                start = time.time()
                result = two_phase_hybrid_solve(instance, timeout=timeout, beam_width=200)
                exec_time = time.time() - start
                
                results.append({
                    'n': n,
                    'm': m,
                    'k': capacity,
                    'execution_time': exec_time,
                    'routes_generated': result.get('routes_generated', 0),
                    'routes_evaluated': result.get('routes_evaluated', 0),
                    'final_capital': result.get('capital'),
                    'timeout': result.get('timeout', False)
                })
                print(f"{exec_time:.3f}s")
            except Exception as e:
                print(f"Error: {e}")
                results.append({
                    'n': n,
                    'm': m,
                    'k': capacity,
                    'execution_time': float('inf'),
                    'routes_generated': 0,
                    'routes_evaluated': 0,
                    'final_capital': None,
                    'timeout': True
                })
    
    return pd.DataFrame(results)

# Run growth analysis
print("Starting growth analysis with m...")
growth_m_df = run_growth_analysis_m(n_values=[30], m_values=[8, 14, 15])
print("\nGrowth analysis complete!")
print(growth_m_df)


In [ ]:
# Save growth analysis results
growth_m_df.to_csv(data_dir / 'growth_analysis_m_with_n_30.csv', index=False)
print("Growth analysis results saved to data/growth_analysis_m_with_n_30.csv")
growth_m_df